In [84]:
import pandas as pd
import yfinance as yf
import numpy as np

## Stock Risk Analysis

The goal of this project is to analyze the risk of holding a group of investment, specifically nine U.S stocks (AAPL, GOOG, COST, CVX, JNJ, MCD, NVDA) and one goal etf (GLD) using a 10-year data from Auguest 29, 2016 to August 28, 2028.

## Step 1: Download Data from Yahoo Finance

In [94]:
tickers = ["AAPL", "GOOG", "COST", "CVX", "JNJ", "JPM", "MCD", "NVDA", "GLD"]

df = yf.download(tickers, start="2016-08-29", end="2026-08-28")

close_prices = df["Close"]

# Piece 1: log returns + drop first NaN row
log_returns = np.log(close_prices / close_prices.shift(1))
log_returns = log_returns.dropna()   # <- method to drop NaN rows

# Piece 2: build summary table
summary = pd.concat([log_returns.mean(), log_returns.std(), log_returns.skew()], axis=1)
summary = summary.rename(columns={0: 'Mean', 1: 'Standard Deviation', 2: 'Skew'})

# Piece 3: annualize mean and std columns only
summary['Mean'] = summary['Mean'] * 252
summary['Standard Deviation'] = summary['Standard Deviation'] * np.sqrt(252)

print(summary)

[*********************100%***********************]  9 of 9 completed

            Mean  Standard Deviation      Skew
Ticker                                        
AAPL    0.256364            0.291216 -0.114652
COST    0.191521            0.221379 -0.533035
CVX     0.109695            0.295029 -1.090135
GLD     0.121161            0.163233 -0.719254
GOOG    0.218445            0.294283 -0.111311
JNJ     0.107112            0.188385 -0.403722
JPM     0.193488            0.273031 -0.092010
MCD     0.105671            0.205393 -0.238373
NVDA    0.502486            0.497605  0.079450


## Stage 2 Self-Check: Which stock is riskiest?

**By volatility alone (annualized std dev):**
NVDA is by far the most volatile (49.8%), followed by CVX, GOOG, and AAPL 
in the 29-30% range. GLD is the calmest by a wide margin (16.4%).

**But volatility alone is misleading.**
Checking mean vs. std across all 9 tickers: every single stock has mean < std.
This is a general feature of daily returns (short-term noise dwarfs long-term 
drift) — not something that distinguishes any one stock from the rest.

**Skew reveals a different risk story.**
GLD has the lowest volatility (16.4%) but a strongly negative skew (-0.725) —
almost as negative as CVX (-1.092), the single most negative in the basket.
Compare this to JPM: nearly double GLD's volatility (27.3%) but a much more 
symmetric skew (-0.092).

**Conclusion:** GLD is the calmest asset day-to-day, but its negative skew means 
its rare "bad days" are disproportionately severe relative to its normal behavior — 
a risk that standard deviation alone doesn't capture. "Riskiest" depends on the 
definition: lowest day-to-day volatility (GLD) is not the same as lowest exposure 
to a severe, asymmetric crash (arguably JPM, given its much more symmetric tail 
despite higher volatility).